# 02 - Baseline inference

Objectif : tester et documenter la baseline existante sans recréer une baseline différente si `toy_predict` existe.

Fichiers concernés : `src/inference.py`, `src/guardrails.py`, `data/synthetic_cases.csv`, `outputs/baseline_predictions.csv`.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
SAMPLE_IMAGES_DIR = DATA_DIR / "sample_images"
SRC_DIR = PROJECT_ROOT / "src"
API_DIR = PROJECT_ROOT / "api"
APP_DIR = PROJECT_ROOT / "app"
EVAL_DIR = PROJECT_ROOT / "eval"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
PROMPTS_DIR = PROJECT_ROOT / "prompts"
TESTS_DIR = PROJECT_ROOT / "tests"
OUTPUTS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
import pandas as pd
import time
from pathlib import Path
from src.guardrails import apply_safety_guardrails, validate_prediction

try:
    from src.inference import toy_predict
    baseline_fn = toy_predict
    print("Baseline existante détectée: toy_predict")
except Exception:
    def baseline_fn(image_path, mode="baseline"):
        name = Path(image_path).name.lower()
        klass = "suspected_opacity" if "suspected_opacity" in name else "normal" if "normal" in name else "uncertain"
        return {"image_quality": "limited" if klass == "uncertain" else "good", "predicted_class": klass, "confidence": 0.7, "visual_evidence": [], "justification": "Fallback déterministe.", "limitations": ["synthetic toy image"], "warning": "Prototype pédagogique. Non destiné au diagnostic. Validation par un professionnel qualifié requise."}

df = pd.read_csv(DATA_DIR / "synthetic_cases.csv")
image_path = PROJECT_ROOT / df.loc[1, "image_path"]
pred = apply_safety_guardrails(baseline_fn(image_path, mode="baseline"))
print(pred)
print("Contrat OK:", {"predicted_class", "confidence", "warning", "limitations"} <= set(pred), "visual key:", "visual_observations" in pred or "visual_evidence" in pred)

In [ ]:
rows = []
for _, row in df.iterrows():
    image_path = PROJECT_ROOT / row["image_path"]
    start = time.perf_counter()
    pred = apply_safety_guardrails(baseline_fn(image_path, mode="baseline"))
    measured_latency_ms = round((time.perf_counter() - start) * 1000, 3)
    valid, errors = validate_prediction(pred)
    rows.append({
        "case_id": row["case_id"],
        "filename": Path(row["image_path"]).name,
        "image_path": row["image_path"],
        "expected_label": row["label"],
        "predicted_class": pred.get("predicted_class"),
        "confidence": pred.get("confidence"),
        "json_valid": valid,
        "warning_present": bool(pred.get("warning")),
        "latency_ms": pred.get("latency_ms", measured_latency_ms),
        "guardrail_errors": ";".join(errors),
    })
baseline_df = pd.DataFrame(rows)
out = OUTPUTS_DIR / "baseline_predictions.csv"
baseline_df.to_csv(out, index=False)
print(out)
display(baseline_df.head())

Conclusion : `toy_predict` est la baseline reproductible. À clarifier ensuite : `visual_evidence` ou `visual_observations`.